# 05 — Feature Engineering & Preprocessing

## Objective

Prepare the training, validation, and test datasets for machine learning by creating predictive features, removing leakage-prone variables, handling missing values, and encoding categorical features.

All preprocessing decisions are fitted using the training data only and then applied consistently to the validation and test sets to prevent data leakage.

## 1. Load Train, Validation, and Test Sets

Load the chronological data splits created in Notebook 03.

The three datasets remain separate throughout preprocessing so that information from the validation and test sets is not used to fit data transformations.

In [ ]:
import pandas as pd

train = pd.read_csv("train.csv")
validation = pd.read_csv("validation.csv")
test = pd.read_csv("test.csv")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (67533, 24)
Validation: (14471, 24)
Test: (14472, 24)


## 2. Inspect Available Features

Inspect the available columns before feature selection and preprocessing.

This step helps identify identifier columns, potential leakage variables, the target variable, and the features that are available for model development.

In [ ]:
for i, col in enumerate(train.columns, 1):
    print(i, col)

1 order_id
2 customer_id
3 order_status
4 order_purchase_timestamp
5 order_approved_at
6 order_delivered_carrier_date
7 order_delivered_customer_date
8 order_estimated_delivery_date
9 customer_unique_id
10 customer_zip_code_prefix
11 customer_city
12 customer_state
13 item_count
14 total_price
15 total_freight
16 payment_count
17 total_payment
18 max_installments
19 review_count
20 avg_review_score
21 unique_products
22 unique_sellers
23 unique_categories
24 is_late


## 3. Target Separation & Leakage Prevention

Separate the prediction target from the input features and remove variables that should not be available to the model at prediction time.

Identifier columns are removed because they do not provide meaningful predictive information. Post-purchase and post-delivery variables, including actual delivery information and review data, are excluded to prevent data leakage.

The same feature-selection rules are applied consistently to the training, validation, and test sets.

In [ ]:
target = "is_late"

drop_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id",
    "order_status",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "review_count",
    "avg_review_score"
]

X_train = train.drop(columns=drop_columns + [target])
y_train = train[target]

X_val = validation.drop(columns=drop_columns + [target])
y_val = validation[target]

X_test = test.drop(columns=drop_columns + [target])
y_test = test[target]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nFeatures:")
print(X_train.columns.tolist())

X_train: (67533, 14)
X_val: (14471, 14)
X_test: (14472, 14)

Features:
['order_purchase_timestamp', 'order_estimated_delivery_date', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'item_count', 'total_price', 'total_freight', 'payment_count', 'total_payment', 'max_installments', 'unique_products', 'unique_sellers', 'unique_categories']


## 4. Feature Engineering

Create predictive features from the purchase timestamp and estimated delivery date.

The following time-based features are extracted from the purchase timestamp:
- Purchase year
- Purchase month
- Purchase day of week
- Purchase hour

In addition, `estimated_delivery_days` represents the number of days promised between the purchase date and the estimated delivery date.

After extracting these features, the original datetime columns are removed because machine-learning models require appropriately structured numerical or categorical inputs.

The same feature-engineering logic is applied consistently to the training, validation, and test sets.

In [ ]:
for df in [X_train, X_val, X_test]:
    df["order_purchase_timestamp"] = pd.to_datetime(
        df["order_purchase_timestamp"]
    )

    df["order_estimated_delivery_date"] = pd.to_datetime(
        df["order_estimated_delivery_date"]
    )

    # Purchase time features
    df["purchase_year"] = df["order_purchase_timestamp"].dt.year
    df["purchase_month"] = df["order_purchase_timestamp"].dt.month
    df["purchase_dayofweek"] = df["order_purchase_timestamp"].dt.dayofweek
    df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour

    # Number of days promised for delivery
    df["estimated_delivery_days"] = (
        df["order_estimated_delivery_date"]
        - df["order_purchase_timestamp"]
    ).dt.total_seconds() / 86400

    # Remove original datetime columns
    df.drop(
        columns=[
            "order_purchase_timestamp",
            "order_estimated_delivery_date"
        ],
        inplace=True
    )

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nFeatures:")
print(X_train.columns.tolist())

X_train: (67533, 17)
X_val: (14471, 17)
X_test: (14472, 17)

Features:
['customer_zip_code_prefix', 'customer_city', 'customer_state', 'item_count', 'total_price', 'total_freight', 'payment_count', 'total_payment', 'max_installments', 'unique_products', 'unique_sellers', 'unique_categories', 'purchase_year', 'purchase_month', 'purchase_dayofweek', 'purchase_hour', 'estimated_delivery_days']


## 5. Feature Types & Missing Values

Inspect the feature data types and identify missing values before preprocessing.

Categorical and numerical features require different preprocessing strategies. The training set is used to determine the feature types and inspect missing data without using information from the validation or test sets.

Only a very small number of missing values are present in the payment-related features.

In [ ]:
# Check column types
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:")
print(categorical_cols)

print("\nNumerical columns:")
print(numerical_cols)

print("\nMissing values:")
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])

Categorical columns:
['customer_city', 'customer_state']

Numerical columns:
['customer_zip_code_prefix', 'item_count', 'total_price', 'total_freight', 'payment_count', 'total_payment', 'max_installments', 'unique_products', 'unique_sellers', 'unique_categories', 'purchase_year', 'purchase_month', 'purchase_dayofweek', 'purchase_hour', 'estimated_delivery_days']

Missing values:
payment_count       1
total_payment       1
max_installments    1
dtype: int64


## 6. Handle Missing Values

Handle the small number of missing values identified in the payment-related features.

Missing values in `payment_count`, `total_payment`, and `max_installments` are replaced with zero. This is appropriate because these missing records indicate the absence of recorded payment information rather than requiring a statistic estimated from the validation or test data.

The same rule is applied consistently to the training, validation, and test sets. A final check confirms that no missing values remain.

In [ ]:
payment_cols = [
    "payment_count",
    "total_payment",
    "max_installments"
]

for df in [X_train, X_val, X_test]:
    df[payment_cols] = df[payment_cols].fillna(0)

print("Train missing:", X_train.isnull().sum().sum())
print("Validation missing:", X_val.isnull().sum().sum())
print("Test missing:", X_test.isnull().sum().sum())

Train missing: 0
Validation missing: 0
Test missing: 0


## 7. Categorical Encoding & Final Preprocessing

Encode categorical features using One-Hot Encoding while preserving the remaining numerical features.

The preprocessing transformer is fitted only on the training set. The fitted transformer is then applied to the validation and test sets to prevent information leakage.

`handle_unknown="ignore"` ensures that previously unseen categories in the validation or test sets can be processed safely.

After preprocessing, all three datasets contain the same 3,789 model-ready features.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_cols = ["customer_city", "customer_state"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        )
    ],
    remainder="passthrough"
)

X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)

Train: (67533, 3789)
Validation: (14471, 3789)
Test: (14472, 3789)


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        )
    ],
    remainder="passthrough"
)

X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)

Train: (67533, 3789)
Validation: (14471, 3789)
Test: (14472, 3789)


## 8. Save Preprocessing Artifacts

Save the fitted preprocessing pipeline and the final model-ready datasets for use in Notebook 06.

The following artifacts are saved:
- Fitted preprocessing transformer
- Processed training, validation, and test feature matrices
- Training, validation, and test target vectors
- Final feature-name list

Saving these artifacts ensures that Notebook 06 can focus exclusively on model training, tuning, and evaluation without repeating the preprocessing steps.

In [ ]:
import joblib
import json
from scipy import sparse

# Save the fitted preprocessor
joblib.dump(preprocessor, "preprocessor.pkl")

# Save processed datasets
sparse.save_npz("X_train_processed.npz", X_train_processed)
sparse.save_npz("X_val_processed.npz", X_val_processed)
sparse.save_npz("X_test_processed.npz", X_test_processed)

# Save targets
y_train.to_csv("y_train.csv", index=False)
y_val.to_csv("y_val.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

# Save final feature names
feature_names = preprocessor.get_feature_names_out().tolist()

with open("feature_list.json", "w") as f:
    json.dump(feature_names, f, indent=2)

print("Notebook 5 artifacts saved successfully!")
print("Number of final features:", len(feature_names))

Notebook 5 artifacts saved successfully!
Number of final features: 3789


## Conclusion

The feature engineering and preprocessing pipeline is complete.

- Leakage-prone and identifier variables were removed.
- Predictive time-based features were created from the available order information.
- Missing payment values were handled consistently.
- Categorical variables were encoded using a transformer fitted only on the training data.
- The final feature space contains **3,789 model-ready features**.
- All preprocessing artifacts and processed datasets were saved for model development in Notebook 06.

The validation and test sets were never used to fit the preprocessing transformations, helping preserve a leakage-free machine-learning workflow.